### Import Dependencies

In [ ]:
import re
from typing import Any, Dict

from guardrails import Guard, OnFailAction

from guardrails.validator_base import (
    FailResult,
    PassResult,
    ValidationResult,
    Validator,
    register_validator
)

### Product ID detection Guardrail

In [ ]:
PRODUCT_ID = re.compile(r"\bB0[A-Z0-9]{8}\b")

In [ ]:
PRODUCT_ID.findall("dasdasd B0B87HSZ8K asdfasfgadsf B0BVRLW1DP")

In [ ]:
@register_validator(name="detect_product_id", data_type="string")
class DetectProductID(Validator):

    def _validate(self, value: Any, metadata: Dict[str, Any] = {}) -> ValidationResult:

        matches = PRODUCT_ID.findall(value)

        if matches:
            return FailResult(
                error_message=f"found {len(matches)} product ids: {matches}",
            )

        return PassResult()

In [ ]:
guard = Guard().use(DetectProductID(on_fail=OnFailAction.NOOP))

In [ ]:
text = "Added 5 units of OYEEICE iPad Air 5th/4th Generation Leather Case (product ID: B09LXP9Z83) to your shopping cart. And one more B09X9RM6FD"

In [ ]:
guardrail_outcome = guard.validate(text)

In [ ]:
guardrail_outcome

In [ ]:
for log in guard.history.last.failed_validations:
    print("Exception: ", log.validation_result.error_message)

In [ ]:
clean_text = "Some random text"

In [ ]:
guardrail_outcome_clean = guard.validate(clean_text)

In [ ]:
guardrail_outcome_clean

### Product ID Redaction Guardrail

In [ ]:
PRODUCT_ID = re.compile(r"\bB0[A-Z0-9]{8}\b")
REPLACEMENT = "[REDACTED..]"

In [ ]:
PRODUCT_ID.sub(REPLACEMENT, "dasdasd B0B87HSZ8K asdfasfgadsf B0BVRLW1DP")

In [ ]:
@register_validator(name="redact_product_id", data_type="string")
class RedactProductID(Validator):

    def _validate(self, value: Any, metadata: Dict[str, Any] = {}) -> ValidationResult:

        matches = PRODUCT_ID.findall(value)

        if matches:
            return FailResult(
                error_message=f"found {len(matches)} product ids: {matches}",
                fix_value=PRODUCT_ID.sub(REPLACEMENT, value)
            )

        return PassResult()

In [ ]:
guard_redact = Guard().use(RedactProductID(on_fail=OnFailAction.FIX))

In [ ]:
outcome_redact = guard_redact.validate(text)

In [ ]:
outcome_redact

In [ ]:
print(f"Output: {outcome_redact.validated_output}")